# Parametric Radiation Hardness Optimization

This notebook sweeps epitaxial layer thickness, bulk doping concentration, and
reverse bias voltage to identify the most radiation-hard 4H-SiC detector
configuration at a target proton fluence.

The sweep uses `radiation_hardness_sweep()` which evaluates CCE at pristine and
irradiated conditions for each configuration, computing CCE retention
($\text{CCE}_{\Phi} / \text{CCE}_0$) as the radiation hardness figure of merit.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.radiation_damage import radiation_hardness_sweep
from src.charge_collection import cce_vs_fluence

plt.rcParams.update({"font.family": "serif", "font.size": 12,
                      "axes.labelsize": 14, "figure.dpi": 150})

In [ ]:
# Sweep parameters
epi_thicknesses = [5e-4, 10e-4, 20e-4, 50e-4]   # 5, 10, 20, 50 um
N_D_bulks = [5e13, 8.5e13, 5e14, 5e15]           # cm^-3
V_biases = [-20, -40, -60, -100]                   # V
target_fluence = 1e13                              # protons/cm^2 (moderate damage)

grid_size = len(epi_thicknesses) * len(N_D_bulks) * len(V_biases)
print(f"{len(epi_thicknesses)} x {len(N_D_bulks)} x {len(V_biases)} = {grid_size} configurations, {grid_size * 2} device solves")
print()
print("Expected runtime: ~15-20 minutes")

In [ ]:
# Run the parametric sweep
df = radiation_hardness_sweep(epi_thicknesses, N_D_bulks, V_biases, target_fluence)
print(f"Sweep complete: {len(df)} configurations")
df.head(10)

In [ ]:
# Top 10 configurations ranked by CCE retention
print("Top 10 Configurations by Radiation Hardness")
print("=" * 80)
top10 = df.head(10).copy()
top10['epi_um'] = top10['epi_thickness_cm'] * 1e4
top10_display = top10[['epi_um', 'N_D_bulk', 'V_bias', 'CCE_pristine', 'CCE_damaged', 'CCE_retention']].copy()
top10_display.columns = ['Epi (um)', 'N_D (cm^-3)', 'V_bias (V)', 'CCE_0', 'CCE_Phi', 'Retention']
top10_display = top10_display.reset_index(drop=True)
top10_display.index += 1
print(top10_display.to_string(float_format=lambda x: f'{x:.4f}' if abs(x) < 100 else f'{x:.2e}'))
print()
best = df.iloc[0]
print(f"Best configuration: epi={best['epi_thickness_cm']*1e4:.0f} um, "
      f"N_D={best['N_D_bulk']:.2e} cm^-3, V_bias={best['V_bias']:.0f} V")
print(f"  CCE retention: {best['CCE_retention']:.4f}")

In [ ]:
# CCE Retention Heatmap: 2x2 subplot (one per epi thickness)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, (epi, ax) in enumerate(zip(epi_thicknesses, axes.flat)):
    subset = df[df['epi_thickness_cm'] == epi]
    pivot = subset.pivot_table(index='N_D_bulk', columns='V_bias', values='CCE_retention')
    
    # Sort index for display
    pivot = pivot.sort_index()
    
    im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto',
                   vmin=df['CCE_retention'].min(), vmax=df['CCE_retention'].max())
    
    # Labels
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{v:.0f}' for v in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{n:.1e}' for n in pivot.index])
    ax.set_xlabel('V_bias (V)')
    ax.set_ylabel(r'N$_D$ (cm$^{-3}$)')
    ax.set_title(f'Epi = {epi*1e4:.0f} um')
    
    # Annotate cells
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                       fontsize=9, fontweight='bold')

fig.suptitle(r'CCE Retention at $10^{13}$ p/cm$^2$ by Device Configuration',
             fontsize=14, fontweight='bold')
fig.colorbar(im, ax=axes, label='CCE Retention', shrink=0.8)
fig.tight_layout(rect=[0, 0, 0.92, 0.96])
fig.savefig('../figures/13a_cce_retention_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Effect of epi thickness at fixed N_D = 8.5e13 (Petringa device)
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']

for V_bias, color in zip(V_biases, colors):
    subset = df[(df['N_D_bulk'] == 8.5e13) & (df['V_bias'] == V_bias)]
    subset = subset.sort_values('epi_thickness_cm')
    ax.plot(subset['epi_thickness_cm'] * 1e4, subset['CCE_retention'],
            'o-', color=color, label=f'V = {V_bias} V', markersize=8, linewidth=2)

ax.set_xlabel(r'Epitaxial thickness ($\mu$m)')
ax.set_ylabel('CCE Retention')
ax.set_title(r'CCE Retention vs Epi Thickness (N$_D$ = 8.5$\times 10^{13}$ cm$^{-3}$)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('../figures/13b_cce_vs_epi_thickness.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Effect of doping at fixed epi = 10 um (Petringa device)
fig, ax = plt.subplots(figsize=(8, 5))

for V_bias, color in zip(V_biases, colors):
    subset = df[(df['epi_thickness_cm'] == 10e-4) & (df['V_bias'] == V_bias)]
    subset = subset.sort_values('N_D_bulk')
    ax.semilogx(subset['N_D_bulk'], subset['CCE_retention'],
                'o-', color=color, label=f'V = {V_bias} V', markersize=8, linewidth=2)

ax.set_xlabel(r'Bulk doping N$_D$ (cm$^{-3}$)')
ax.set_ylabel('CCE Retention')
ax.set_title(r'CCE Retention vs Doping (Epi = 10 $\mu$m)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('../figures/13c_cce_vs_doping.png', dpi=300, bbox_inches='tight')
plt.show()

## Discussion and Recommendations

### Key Findings

**Epitaxial thickness** has the largest effect on radiation hardness:
- Thinner detectors maintain higher CCE retention because the depletion region
  (relative to total thickness) is better preserved.
- However, thinner detectors collect less total charge in pristine conditions.

**Bulk doping** affects radiation hardness through two competing mechanisms:
- Higher doping provides more margin before full compensation ($\Phi_{\text{crit}}$
  increases), maintaining the electric field longer.
- However, higher doping reduces the pristine depletion width at a given bias,
  which may reduce pristine CCE.

**Bias voltage** directly recovers CCE at all fluences:
- Higher reverse bias extends the depletion region into radiation-damaged material.
- Trade-off: higher bias increases dark current and may exceed breakdown limits.

### Optimal Configuration

The optimal configuration depends on the operational priority:
- **Maximum radiation hardness**: Thin epi + high doping + high bias
- **Best pristine performance**: Thick epi + moderate doping + moderate bias
- **Balanced**: The Petringa baseline (10 um, 8.5e13 cm$^{-3}$, -40 V)
  provides a good compromise between pristine CCE and radiation tolerance.

## Summary

Key takeaways from the parametric radiation hardness study:

- **Epitaxial thickness** is the dominant parameter for radiation hardness;
  thinner detectors retain more CCE after irradiation.
- **Doping concentration** provides a secondary effect: higher doping delays
  full carrier compensation but may reduce pristine collection.
- **Bias voltage** is the most operationally accessible parameter for recovering
  CCE after irradiation, at the cost of increased dark current.
- The **Petringa baseline device** (10 um epi, 8.5e13 cm$^{-3}$, -40 V) represents
  a balanced design point.
- For applications requiring extreme radiation tolerance, **5 um epi with high
  doping and -60 to -100 V bias** should be considered.